# Semana 2 — Transformación del Dataset para Machine Learning

**Objetivo:** Transformar el dataset original (formato largo, 2000–2022) a un formato utilizable para regresión.

**Problema a resolver:** Estimar el porcentaje de uso de Internet por país, año y grupo etario.

---

**Archivo principal:** `02_semana2_transformacion.ipynb`
**Dataset transformado:** `outputs/datos_transformados_semana2.csv`

## 1. Diseño del Dataset Transformado

### 1. Análisis del Formato Original

El dataset original está en formato **largo**, donde cada fila representa una observación de país, año y grupo etario. Esta estructura es ideal para exploración visual pero **no es suficiente para regresión** por las siguientes razones:

- **Carencia de variables temporales explícitas:** El año es una dimensión de la tabla, no una variable numérica que capture tendencia.
- **Sin estructura tabular directa:** Para regresión supervisada necesitamos variables independientes en columnas y variable objetivo aislada.
- **Columnas redundantes:** `indicator`, `unit`, `notes_ids`, `source_id` tienen valores fijos o vacíos sin aportar señal predictiva.
- **Panel desbalanceado:** Requiere limpieza (valores faltantes, período confiable desde 2016).

**Recomendación de Semana 1:** Pivotaje a ancho + codificación categórica + variables temporales.

### 2. Clasificación de Columnas del Dataset Original

| Columna | Tipo | Decisión | Justificación |
|---------|------|----------|---------------|
| `indicator` | string (fijo) | **ELIMINAR** | Valor constante; sin variabilidad para ML |
| `País__ESTANDAR` | string | **CONSERVAR** | Dimensión clave del problema; diferencia importante entre países |
| `Grupos etarios Uso Internet` | string | **CONSERVAR** (sin "Total") | Dimensión clave; modelar por grupo etario específico |
| `Años__ESTANDAR` | numeric | **CONSERVAR + TRANSFORMAR** | Base temporal; crear `years_since_2016` para capturar tendencia |
| `value` | numeric (0–100) | **CONSERVAR** | Variable objetivo; porcentaje de usuarios de Internet |
| `unit` | string (fijo) | **ELIMINAR** | Valor constante; sin valor predictivo |
| `notes_ids` | string (vacío) | **ELIMINAR** | Mayormente faltante; sin valor para modelo |
| `source_id` | numeric (fijo) | **ELIMINAR** | Identificador de fuente; no añade información analítica |

### 3. Propuesta de Estructura Final del Dataset

#### Unidad de análisis
**Cada fila representa una observación disponible de país, año y grupo etario con su tasa de uso de Internet.**

**Período de trabajo:** 2016–2022
**Filtros aplicados:** se excluyó el grupo "Total" y solo se conservaron los grupos etarios específicos.

#### Columnas del CSV transformado

| Columna | Tipo | Rango/Valores | Descripción |
|---------|------|---------------|-------------|
| `pais` | string | 13 países únicos | Nombre estandarizado del país |
| `anio` | int | 2016–2022 | Año del registro |
| `years_since_2016` | int | 0–6 | Años transcurridos desde 2016 |
| `porcentaje_internet` | int | 0–100 | Variable objetivo |
| `grupo_*` | int | 0 o 1 | Variables dummy del grupo etario para usar el dataset en ML |

#### Tamaño esperado
- Panel completo teórico: **13 países × 7 años × 5 grupos = 455 combinaciones posibles**
- Observaciones realmente disponibles después del filtrado: **340 filas**
- Combinaciones ausentes: **115**
- Estructura final exportada: **340 × 9** (4 columnas base + 5 dummies)


### 4. Justificación del Diseño Propuesto

#### Por qué esta estructura es mejor que el original

1. **Tabular y lista para ML**
   - Cada fila representa una observación independiente.
   - Las variables categóricas relevantes quedaron codificadas en columnas numéricas.
   - Se eliminaron columnas redundantes o sin señal predictiva.

2. **Captura de tendencia temporal**
   - `years_since_2016` permite que el modelo aprenda la evolución temporal sin usar el año como texto.
   - El período 2016–2022 concentra la parte más confiable del panel.

3. **Alineación con el problema**
   - El objetivo es una regresión sobre el porcentaje de uso de Internet.
   - La combinación de país, año y grupo etario se conserva como unidad analítica.
   - El grupo etario queda representado por variables dummy para que sea utilizable en modelos supervisados.

4. **Reducción de ruido estructural**
   - Se eliminaron `indicator`, `unit`, `notes_ids` y `source_id` porque no aportan variación útil.
   - Se descartó el grupo `Total` para evitar mezclarlo con grupos etarios específicos.

5. **Diseño coherente con la calidad del dato**
   - El panel es incompleto: hay 115 combinaciones faltantes.
   - No se imputaron esas filas porque la ausencia también refleja la cobertura real del dataset.
   - El archivo final conserva solo observaciones reales y verificables.

#### Decisiones clave registradas

| Decisión | Valor | Justificación |
|----------|-------|---------------|
| Unidad de análisis | país-año-grupo | Estructura requerida para regresión supervisada |
| Período | 2016–2022 | Mejor cobertura y consistencia que el período completo |
| Filtro grupo "Total" | Eliminar | Solo grupos etarios específicos |
| Variables derivadas | `years_since_2016` | Captura tendencia temporal numérica |
| Codificación categórica | Variables dummy del grupo etario | Hace el archivo apto para ML |
| Columnas eliminadas | `indicator`, `unit`, `notes_ids`, `source_id` | Sin valor predictivo |


## 2. Transformación del Dataset

In [1]:
import pandas as pd
import numpy as np
import os

df_original = pd.read_csv('../data/datos.csv', sep=';', encoding='latin1')
print(f"Dataset original: {df_original.shape[0]} filas x {df_original.shape[1]} columnas")
print(f"Columnas: {list(df_original.columns)}")
df_original.head()

Dataset original: 870 filas x 8 columnas
Columnas: ['indicator', 'País__ESTANDAR', 'Grupos etarios Uso Internet', 'Años__ESTANDAR', 'value', 'unit', 'notes_ids', 'source_id']


,indicator,País__ESTANDAR,Grupos etarios Uso Internet,Años__ESTANDAR,value,unit,notes_ids,source_id
0,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2016,76,Porcentaje sobre el total de personas en cada ...,NaN,9353
1,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2017,76,Porcentaje sobre el total de personas en cada ...,NaN,9353
2,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2018,79,Porcentaje sobre el total de personas en cada ...,NaN,9353
3,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2019,79,Porcentaje sobre el total de personas en cada ...,NaN,9353
4,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2020,88,Porcentaje sobre el total de personas en cada ...,NaN,9353


In [2]:
col_pais = df_original.columns[1]   # País__ESTANDAR
col_anios = df_original.columns[3]  # Años__ESTANDAR
col_grupo = 'Grupos etarios Uso Internet'

# Filtrar período 2016–2022 y excluir grupo "Total"
df_filtrado = df_original[
    (df_original[col_anios] >= 2016) &
    (df_original[col_anios] <= 2022) &
    (df_original[col_grupo] != 'Total')
].copy()

print(f"Filas después de filtrar: {df_filtrado.shape[0]}")
print(f"Países únicos: {df_filtrado[col_pais].nunique()}")
print(f"Años únicos: {sorted(df_filtrado[col_anios].unique())}")
print(f"Grupos etarios: {list(df_filtrado[col_grupo].unique())}")

Filas después de filtrar: 340
Países únicos: 13
Años únicos: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Grupos etarios: ['edad de medicion a 17 años', '18 a 25 años de edad', '26 a 50 años de edad', '51 a 65 años', '66 años en adelante']


In [3]:
# Selección, renombrado y creación de variable temporal
df_transformado = df_filtrado.rename(columns={
    col_pais: 'pais',
    col_anios: 'anio',
    col_grupo: 'grupo_etario',
    'value': 'porcentaje_internet'
})[['pais', 'anio', 'grupo_etario', 'porcentaje_internet']].copy()

orden_grupos = [
    'edad de medicion a 17 años',
    '18 a 25 años de edad',
    '26 a 50 años de edad',
    '51 a 65 años',
    '66 años en adelante'
]
df_transformado['grupo_etario'] = pd.Categorical(
    df_transformado['grupo_etario'],
    categories=orden_grupos,
    ordered=True
)
df_transformado = df_transformado.sort_values(['pais', 'anio', 'grupo_etario']).reset_index(drop=True)

df_transformado['years_since_2016'] = df_transformado['anio'] - 2016
df_transformado['porcentaje_internet'] = df_transformado['porcentaje_internet'].astype(int)

# Codificación de variables dummy para grupo_etario
# Se normalizan los nombres para que el CSV quede estable y sin caracteres acentuados en los encabezados
dummies = pd.get_dummies(df_transformado['grupo_etario'], prefix='grupo', dtype=int)
dummies.columns = (
    dummies.columns
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
)
orden_dummies = [
    'grupo_18 a 25 anos de edad',
    'grupo_26 a 50 anos de edad',
    'grupo_51 a 65 anos',
    'grupo_66 anos en adelante',
    'grupo_edad de medicion a 17 anos'
]
dummies = dummies[orden_dummies]
df_ml = pd.concat([df_transformado, dummies], axis=1)

# El CSV final no conserva la columna textual del grupo etario; queda codificada en dummies
df_final = df_ml.drop(columns=['grupo_etario']).copy()
columnas_finales = ['pais', 'anio', 'years_since_2016', 'porcentaje_internet'] + orden_dummies
df_final = df_final[columnas_finales]

df_ml['grupo_etario'] = df_ml['grupo_etario'].astype(str)

print(f"Dataset intermedio: {df_ml.shape[0]} filas x {df_ml.shape[1]} columnas")
print(f"Dataset final listo para exportar: {df_final.shape[0]} filas x {df_final.shape[1]} columnas")
print(f"Columnas finales: {list(df_final.columns)}")
df_final.head()


Dataset intermedio: 340 filas x 10 columnas
Dataset final listo para exportar: 340 filas x 9 columnas
Columnas finales: ['pais', 'anio', 'years_since_2016', 'porcentaje_internet', 'grupo_18 a 25 anos de edad', 'grupo_26 a 50 anos de edad', 'grupo_51 a 65 anos', 'grupo_66 anos en adelante', 'grupo_edad de medicion a 17 anos']


,pais,anio,years_since_2016,porcentaje_internet,grupo_18 a 25 anos de edad,grupo_26 a 50 anos de edad,grupo_51 a 65 anos,grupo_66 anos en adelante,grupo_edad de medicion a 17 anos
0,Argentina,2016,0,76,0,0,0,0,1
1,Argentina,2016,0,86,1,0,0,0,0
2,Argentina,2016,0,82,0,1,0,0,0
3,Argentina,2016,0,61,0,0,1,0,0
4,Argentina,2016,0,29,0,0,0,1,0


In [4]:
# Validación rápida antes de exportar
print("Duplicados en el dataset final:", df_final.duplicated().sum())
print("Nulos explícitos en el dataset final:\n", df_final.isnull().sum().to_string())
print("\nRango porcentaje_internet:", df_final['porcentaje_internet'].min(), "–", df_final['porcentaje_internet'].max())
print("Países:", sorted(df_final['pais'].unique()))


Duplicados en el dataset final: 0
Nulos explícitos en el dataset final:
 pais                                0
anio                                0
years_since_2016                    0
porcentaje_internet                 0
grupo_18 a 25 anos de edad          0
grupo_26 a 50 anos de edad          0
grupo_51 a 65 anos                  0
grupo_66 anos en adelante           0
grupo_edad de medicion a 17 anos    0

Rango porcentaje_internet: 3 – 97
Países: ['Argentina', 'Bolivia (Estado Plurinacional de)', 'Chile', 'Colombia', 'Costa Rica', 'Ecuador', 'El Salvador', 'Honduras', 'México', 'Panamá', 'Paraguay', 'Perú', 'Uruguay']


In [5]:
os.makedirs('../outputs', exist_ok=True)
df_final.to_csv('../outputs/datos_transformados_semana2.csv', index=False, encoding='utf-8')
print("CSV exportado: outputs/datos_transformados_semana2.csv")


CSV exportado: outputs/datos_transformados_semana2.csv


## 3. Validación de Calidad del Dataset Final

Se revisan valores faltantes (explícitos e implícitos), duplicados, rangos, tipos de dato y coherencia general del dataset transformado.

### 3.1 Valores Faltantes Explícitos: Original vs. Transformado

In [6]:
# Nulos explícitos en el dataset original
print("=== Nulos en dataset original ===")
print(df_original.isnull().sum().to_string())

print("\n=== Nulos en dataset final exportable ===")
print(df_final.isnull().sum().to_string())


=== Nulos en dataset original ===
indicator                        0
País__ESTANDAR                   0
Grupos etarios Uso Internet      0
Años__ESTANDAR                   0
value                            0
unit                             0
notes_ids                      870
source_id                        0

=== Nulos en dataset final exportable ===
pais                                0
anio                                0
years_since_2016                    0
porcentaje_internet                 0
grupo_18 a 25 anos de edad          0
grupo_26 a 50 anos de edad          0
grupo_51 a 65 anos                  0
grupo_66 anos en adelante           0
grupo_edad de medicion a 17 anos    0


### 3.2 Valores Faltantes Implícitos (Combinaciones Ausentes)

El dataset transformado no tiene nulos explícitos, pero al verificar todas las combinaciones posibles de país × año × grupo etario se detectan **combinaciones faltantes** (filas inexistentes). Estas representan observaciones donde no hay datos disponibles y no generan una fila con NaN, sino que directamente no existen.

In [8]:
# Construir índice completo de todas las combinaciones posibles
paises = df_transformado['pais'].unique()
anios = range(2016, 2023)
grupos = df_transformado['grupo_etario'].unique()

indice_completo = pd.MultiIndex.from_product(
    [paises, anios, grupos],
    names=['pais', 'anio', 'grupo_etario']
)
total_posible = len(indice_completo)

indice_existente = df_transformado.set_index(['pais', 'anio', 'grupo_etario']).index
faltantes = indice_completo.difference(indice_existente)
faltantes_df = pd.DataFrame(list(faltantes), columns=['pais', 'anio', 'grupo_etario'])

print(f"Combinaciones posibles: {total_posible}")
print(f"Combinaciones presentes: {len(df_transformado)}")
print(f"Combinaciones faltantes: {len(faltantes_df)} ({len(faltantes_df)/total_posible*100:.1f}%)")
print("\nFaltantes por país:")
print(faltantes_df.groupby('pais').size().sort_values(ascending=False).to_string())
print("\nFaltantes por año:")
print(faltantes_df.groupby('anio').size().to_string())


Combinaciones posibles: 455
Combinaciones presentes: 340
Combinaciones faltantes: 115 (25.3%)

Faltantes por país:
pais
Chile                                30
Honduras                             20
El Salvador                          15
Ecuador                              15
Uruguay                              15
Panamá                               10
Colombia                              5
Bolivia (Estado Plurinacional de)     5

Faltantes por año:
anio
2016     5
2017    10
2018     5
2019     5
2020    30
2021    30
2022    30


In [9]:
# Matriz de cobertura: filas por país y año
cobertura = df_transformado.groupby(['pais', 'anio']).size().unstack(fill_value=0)
print("Cobertura por país y año (número de grupos etarios con datos):")
print(cobertura.to_string())


Cobertura por país y año (número de grupos etarios con datos):
anio                               2016  2017  2018  2019  2020  2021  2022
pais                                                                       
Argentina                             5     5     5     5     5     5     5
Bolivia (Estado Plurinacional de)     5     5     5     5     5     5     0
Chile                                 0     5     0     0     0     0     0
Colombia                              5     0     5     5     5     5     5
Costa Rica                            5     5     5     5     5     5     5
Ecuador                               5     5     5     5     0     0     0
El Salvador                           5     5     5     5     0     0     0
Honduras                              5     0     5     5     0     0     0
México                                5     5     5     5     5     5     5
Panamá                                5     5     5     5     0     0     5
Paraguay                 

### 3.3 Duplicados

In [10]:
dup_filas = df_ml.duplicated().sum()
dup_clave = df_ml.duplicated(['pais', 'anio', 'grupo_etario']).sum()

print(f"Filas duplicadas (completas): {dup_filas}")
print(f"Duplicados por clave (pais, anio, grupo_etario): {dup_clave}")


Filas duplicadas (completas): 0
Duplicados por clave (pais, anio, grupo_etario): 0


### 3.4 Valores Fuera de Rango e Inconsistencias

In [11]:
fuera_rango = ((df_ml['porcentaje_internet'] < 0) | (df_ml['porcentaje_internet'] > 100)).sum()
print(f"Valores fuera de rango [0, 100] en porcentaje_internet: {fuera_rango}")
print(f"Rango observado: {df_ml['porcentaje_internet'].min():.1f} – {df_ml['porcentaje_internet'].max():.1f}")
print(f"\nEstadísticas descriptivas de porcentaje_internet:")
print(df_ml['porcentaje_internet'].describe().to_string())

anios_invalidos = (~df_ml['anio'].between(2016, 2022)).sum()
ysr_invalidos = (~df_ml['years_since_2016'].between(0, 6)).sum()
print(f"\nAños fuera de 2016–2022: {anios_invalidos}")
print(f"years_since_2016 fuera de 0–6: {ysr_invalidos}")
print(f"\nGrupos etarios únicos:\n{df_ml['grupo_etario'].value_counts().to_string()}")


Valores fuera de rango [0, 100] en porcentaje_internet: 0
Rango observado: 3.0 – 97.0

Estadísticas descriptivas de porcentaje_internet:
count    340.000000
mean      58.911765
std       25.896399
min        3.000000
25%       39.000000
50%       64.000000
75%       80.250000
max       97.000000

Años fuera de 2016–2022: 0
years_since_2016 fuera de 0–6: 0

Grupos etarios únicos:
grupo_etario
edad de medicion a 17 años    68
18 a 25 años de edad          68
26 a 50 años de edad          68
51 a 65 años                  68
66 años en adelante           68


### 3.5 Tipos de Dato

In [12]:
print("Tipos de dato en el dataset transformado:")
print(df_ml.dtypes.to_string())
print("\nVerificación esperada:")
esperados = {
    'pais': 'str', 'anio': 'int64', 'grupo_etario': 'str',
    'porcentaje_internet': 'int64', 'years_since_2016': 'int64'
}
for col, tipo_esp in esperados.items():
    tipo_real = str(df_ml[col].dtype)
    estado = "OK" if tipo_real == tipo_esp else f"REVISAR (es {tipo_real})"
    print(f"  {col}: esperado {tipo_esp} — {estado}")


Tipos de dato en el dataset transformado:
pais                                  str
anio                                int64
grupo_etario                          str
porcentaje_internet                 int64
years_since_2016                    int64
grupo_18 a 25 anos de edad          int64
grupo_26 a 50 anos de edad          int64
grupo_51 a 65 anos                  int64
grupo_66 anos en adelante           int64
grupo_edad de medicion a 17 anos    int64

Verificación esperada:
  pais: esperado str — OK
  anio: esperado int64 — OK
  grupo_etario: esperado str — OK
  porcentaje_internet: esperado int64 — OK
  years_since_2016: esperado int64 — OK


### 3.6 Coherencia del CSV Final

In [13]:
df_csv = pd.read_csv('../outputs/datos_transformados_semana2.csv', encoding='utf-8')
print(f"Filas en CSV: {df_csv.shape[0]}")
print(f"Columnas en CSV: {df_csv.shape[1]}")
print(f"Columnas: {list(df_csv.columns)}")
print(f"\nNulos en CSV:\n{df_csv.isnull().sum().to_string()}")
print(f"\nEl CSV coincide con el dataset final: {df_csv.shape == df_final.shape}")
print(f"El CSV coincide en columnas: {list(df_csv.columns) == list(df_final.columns)}")
print(f"\nPrimeras filas:")
df_csv.head()


Filas en CSV: 340
Columnas en CSV: 9
Columnas: ['pais', 'anio', 'years_since_2016', 'porcentaje_internet', 'grupo_18 a 25 anos de edad', 'grupo_26 a 50 anos de edad', 'grupo_51 a 65 anos', 'grupo_66 anos en adelante', 'grupo_edad de medicion a 17 anos']

Nulos en CSV:
pais                                0
anio                                0
years_since_2016                    0
porcentaje_internet                 0
grupo_18 a 25 anos de edad          0
grupo_26 a 50 anos de edad          0
grupo_51 a 65 anos                  0
grupo_66 anos en adelante           0
grupo_edad de medicion a 17 anos    0

El CSV coincide con el dataset final: True
El CSV coincide en columnas: True

Primeras filas:


,pais,anio,years_since_2016,porcentaje_internet,grupo_18 a 25 anos de edad,grupo_26 a 50 anos de edad,grupo_51 a 65 anos,grupo_66 anos en adelante,grupo_edad de medicion a 17 anos
0,Argentina,2016,0,76,0,0,0,0,1
1,Argentina,2016,0,86,1,0,0,0,0
2,Argentina,2016,0,82,0,1,0,0,0
3,Argentina,2016,0,61,0,0,1,0,0
4,Argentina,2016,0,29,0,0,0,1,0


### 3.7 Resumen de Problemas y Limitaciones

**Problemas corregidos durante la transformación:**

- `notes_ids`: columna 100% nula en el dataset original; se eliminó.
- `indicator`, `unit` y `source_id`: columnas constantes sin aporte predictivo; se eliminaron.
- Grupo `Total`: se excluyó para trabajar solo con grupos etarios específicos.
- Grupo etario textual: se convirtió en variables dummy para dejar el archivo listo para ML.

**Problemas identificados que quedan como limitación:**

- **115 combinaciones faltantes (25.3% del panel):** el dataset no cubre todas las combinaciones país × año × grupo etario posibles. Los registros ausentes se concentran sobre todo entre 2020 y 2022, lo que sugiere una pérdida de cobertura en los años recientes.
- **Panel desbalanceado:** la heterogeneidad en cobertura temporal por país puede introducir sesgo en los modelos.
- **Periodo reducido a 2016–2022:** se priorizó consistencia y cobertura sobre amplitud histórica.
- **Variable objetivo en enteros:** `porcentaje_internet` proviene del dato original sin decimales, por lo que la resolución del objetivo es discreta aunque el problema sea de regresión.


## 4. Integración Final y Cierre

La transformación queda consistente con el problema seleccionado en Semana 1: regresión para estimar el porcentaje de uso de Internet por país, año y grupo etario.

### Verificación global

- El archivo original estaba en formato largo y tenía columnas de contexto que no aportaban señal predictiva directa.
- El dataset final conserva solo observaciones reales del período 2016–2022.
- El grupo etario quedó codificado en variables dummy para que el CSV sea utilizable en modelos de machine learning.
- El archivo exportado no conserva el formato original y sí responde al objetivo analítico definido.

### Entrega lista para Semana 3

- Notebook principal: `notebooks/02_semana2_transformacion.ipynb`
- Dataset final: `outputs/datos_transformados_semana2.csv`
- Estructura final: 340 filas y 9 columnas
- Limitación principal: 115 combinaciones país-año-grupo siguen ausentes porque no existían en la fuente original.

Con esto quedan integradas la definición del dataset, la transformación, la validación de calidad y la exportación final.
